# 01 — Download GSS cumulative data

Downloads the GSS 1972–2024 cumulative cross-sectional file (Stata format,
Release 3a) from NORC into `../data/raw/gss/`, extracts the handful of
variables needed for the Jewish identity trend analysis, and saves them to
`../data/derived/gss_jewish_identity.parquet`.

Variables kept:

| variable | label |
|----------|-------|
| `year`, `id` | survey year, respondent ID |
| `relig` | R's religious preference |
| `relig16` | religion in which raised (at age 16) |
| `jew`, `jew16` | Jewish denomination, current / at 16 |
| `wtssps` | person post-stratification weight (defined for **all** years, including 2021+) |


In [ ]:
import os
import zipfile
import urllib.request

DATA = os.path.abspath(os.path.join('..', 'data'))
RAW = os.path.join(DATA, 'raw', 'gss')
DERIVED = os.path.join(DATA, 'derived')
os.makedirs(RAW, exist_ok=True)
os.makedirs(DERIVED, exist_ok=True)

URL = 'https://gss.norc.org/content/dam/gss/get-the-data/documents/stata/GSS_stata.zip'
zip_path = os.path.join(RAW, 'GSS_stata.zip')
dta_path = os.path.join(RAW, 'GSS_stata', 'gss7224_r3a.dta')

if not os.path.exists(dta_path):
    if not os.path.exists(zip_path):
        print('downloading', URL)
        urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW)
print(f'{os.path.getsize(dta_path)/1e6:.0f} MB', dta_path)

In [ ]:
import warnings
import pandas as pd

warnings.filterwarnings('ignore', message='.*could not be decoded.*')

cols = ['year', 'id', 'relig', 'relig16', 'jew', 'jew16', 'wtssps']
df = pd.read_stata(dta_path, columns=cols, convert_categoricals=True)
df['year'] = df['year'].astype(int)
print(df.shape)
df.head()

In [ ]:
out_path = os.path.join(DERIVED, 'gss_jewish_identity.parquet')
df.to_parquet(out_path)

# quick sanity check: coverage and raw Jewish counts by decade
chk = df.assign(decade=df.year // 10 * 10).groupby('decade').agg(
    n=('id', 'size'),
    n_jewish=('relig', lambda s: (s == 'jewish').sum()),
    n_raised_jewish=('relig16', lambda s: (s == 'jewish').sum()),
    wt_missing=('wtssps', lambda s: s.isna().sum()),
)
chk